this is the script to train a diffusion model.

In [1]:
# import
import sys 
sys.path.append('/workspace/Documents')
import os
import torch
import numpy as np
import Diffusion_for_CT_motion.diffusion_models.conditional_diffusion_3D as ddpm_3D
import Diffusion_for_CT_motion.diffusion_models.conditional_EDM_3D as edm
import Diffusion_for_CT_motion.utils.functions_collection as ff
import Diffusion_for_CT_motion.utils.Build_list as Build_list
import Diffusion_for_CT_motion.utils.Generator as Generator

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/Documents/Diffusion_models/denoising_diffusion_pytorch/denoising_diffusion_pytorch/standard_diffusion.py:773: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/workspace/Documents/Diffusion_models/denoising_diffusion_pytorch/denoising_diffusion_pytorch/conditional_diffusion.py:950: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/workspace/Documents/Diffusion_models/denoising_diffusion_pytorch/denoising_diffusion_pytorch/conditional_diffusion_3D.py:865: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `tor

#### step 1: set the trial name and pre-trained model path

In [2]:
trial_name = 'diffusion_model'
pre_trained_model = None #  or path of the pre-trained model
start_step = 0 # if new training, start step = 0, if continue, start_step = None

#### step 2: define the training  and validation cohort
here just as an example, we use the same one case for training and validation.

The cohort information is saved in a spreadsheet "patient_list.xlsx", which you can easily generate with your own data paths.

In [3]:
# define train
build_sheet =  Build_list.Build('patient_list.xlsx')  # this is data path for training data
_,_, x0_list_train, condition_list_train= build_sheet.__build__(batch_list = [0]) 

# define val
_,_, x0_list_val, condition_list_val= build_sheet.__build__(batch_list = [0])

#### step 3: set some default parameter

In [4]:
# set default, don't change unless necessary
image_size_3D = [256,256,50]  # size of 3D head CT
patch_size = 128  # size of 3D patch when using patch-wise training

# don't change the following, 
histogram_equalization = True # alreayd set True
# these two are used for histogram equalization
bins = np.load('bins.npy') # provide these two files in the repo
bins_mapped = np.load('bins_mapped.npy')

# for data normalization 
background_cutoff = -1000 
maximum_cutoff = 2000
normalize_factor = 'equation'

#### step 4: define the U-Net and the EDM

In [5]:
# don't change unless necessary
model = ddpm_3D.Unet3D(
    init_dim = 64,
    channels = 1, 
    dim_mults = (1, 2, 4, 8),
    flash_attn = False,
    conditional_diffusion = True,
    full_attn = (None, None, False, True),)
 
diffusion_model = edm.EDM(
    model,
    image_size = [patch_size, patch_size, image_size_3D[-1]], # 3D patch-wise training
    num_sample_steps = 50, # default
    clip_or_not = False,)

#### step 5: define data generator for training and validation

In [6]:
generator_train = Generator.Dataset_dual_patch( 
    x0_list_train,
    condition_list_train,

    image_size_3D = image_size_3D,
    slice_start = [2,12],
    slice_num = 50,

    patch_size = patch_size,
    patch_stride = patch_size,
    original_patch_num = 0,
    random_sampled_patch_num = 4,
   
    histogram_equalization = histogram_equalization, 
    bins = bins,
    bins_mapped = bins_mapped,

    background_cutoff = background_cutoff, 
    maximum_cutoff = maximum_cutoff,
    normalize_factor = normalize_factor,

    shuffle = True,augment = True, augment_frequency = 0.5,)


generator_val = Generator.Dataset_dual_patch(
    x0_list_val,
    condition_list_val,

    image_size_3D = image_size_3D,
    slice_start = [10,20],
    slice_num = 20, # fewer slices since it's validating on the entire image, we need to consider GPU memory

    patch_size = 256, # validate on the entire image
    patch_stride = 1,
    original_patch_num = 1,
    random_sampled_patch_num = 0,

    histogram_equalization = histogram_equalization, 
    bins = bins,
    bins_mapped = bins_mapped,
    
    background_cutoff = background_cutoff, 
    maximum_cutoff = maximum_cutoff,
    normalize_factor = normalize_factor,)

#### step 6: train the model

In [7]:
trainer = edm.Trainer(
    diffusion_model= diffusion_model,
    batch_size = 1,
    generator_train = generator_train,
    generator_val = generator_val,

    train_num_steps = 200, # total training epochs
    save_folder = os.path.join('/mnt/camca_NAS/diffusion_ct_motion/models', trial_name, 'models'),
   
    train_lr = 1e-4,
    train_lr_decay_every = 100, 
    save_models_every = 1,
    validation_every = 1,)

trainer.train(pre_trained_model=pre_trained_model, start_step= start_step)

  0%|          | 0/200 [00:00<?, ?it/s]

training epoch:  1
learning rate:  0.0001
current slice range:  [10, 60]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)
current slice range:  [3, 53]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)
current slice range:  [11, 61]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)
current slice range:  [5, 55]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)


average loss: 1.7136:   0%|          | 0/200 [00:05<?, ?it/s]

validation at step:  1
current slice range:  [13, 33]
x0_image_data.shape, condition_image_data.shape (256, 256, 20) (256, 256, 20)


average loss: 1.7136:   0%|          | 1/200 [00:21<1:10:21, 21.21s/it]

validation loss:  0.9264136552810669
now run on_epoch_end function
now run on_epoch_end function
training epoch:  2
learning rate:  0.0001
current slice range:  [8, 58]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)
current slice range:  [6, 56]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)
current slice range:  [12, 62]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)
current slice range:  [2, 52]
x0_image_data.shape, condition_image_data.shape (128, 128, 50) (128, 128, 50)


average loss: 1.2629:   0%|          | 1/200 [00:38<2:09:01, 38.90s/it]


KeyboardInterrupt: 